In [ ]:
import sys
import os
from pathlib import Path  # noqa: F401

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

import zipfile  # noqa: E402
import numpy as np  # noqa: E402
import numpy.lib.format as npformat  # noqa: E402
import pandas as pd  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
import seaborn as sns  # noqa: E402
import mne  # noqa: E402
from scipy.stats import zscore  # noqa: E402
from sklearn.decomposition import PCA  # noqa: E402

from src.preprocessing.pipeline import DatasetHandler  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    MusicTypeVariants,
    ExperimentNames,
    CoordinateSystems,
    PreprocessedDataVariants,
)
from src.definitions.constants import ProjectPaths  # noqa: E402

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
mne.set_log_level("ERROR")
print("Setup complete.")

# ASSR Raw Voltage — Channel-PCA Component (topomap + time course)

Raw-voltage analog of `assr_wavelet_pca_analysis.ipynb`. Instead of wavelet power,
this reduces the stimulus-locked **evoked voltage** to a single **spatial
component per participant** by collapsing the channel dimension with a **PCA over
channels**.

Pipeline (per subject, on the **Placebo** condition by project convention):

1. **Trial average.** Epoch the raw signal around every `fam+` onset and average
   over stimuli — the phase-locked *evoked* response `(n_channels, win)`. The
   epoch keeps a fixed pad before onset and a post-onset length equal to the
   **shortest inter-onset interval**, so no epoch overlaps the next stimulus.
2. **Z-score per channel** (over the whole recording) so every electrode has unit
   variance and contributes **equally** to the PCA. Without this the channel PCA
   is a covariance-PCA dominated by the highest-variance electrodes; z-scoring
   turns it into a correlation-PCA with equal electrode influence. The trade-off:
   the component score is then in normalized units, not µV.
3. **Channel PCA.** Treat each **time sample** as an observation and each channel
   as a variable — matrix `X` of shape `(win, n_channels)` — and fit PCA. The
   **first component** gives two things: its **loading vector** `(n_channels,)` is
   the component's scalp **topography**, and its **score** `(win,)` is the
   component's **time course**.
4. **Polarity alignment.** PCA component signs are arbitrary, so some participants
   come out flipped. Every subject's loading is aligned to a common template
   (iteratively-refined group mean) and the global orientation anchored to a
   reference channel, giving **one consistent polarity across participants**; the
   score is flipped with the loading so topography and time course stay consistent.
5. **Plot** the per-participant PC1 topomap and PC1 time course.

Z-scoring is per channel against the **whole recording** (not per epoch), matching
the wavelet notebook's normalization rationale. The stimulus alignment (see
`00-preprocessing/stimulus_alignment.ipynb`) puts every onset at the same sample
index in all participants, so one shared onsets array serves every subject.

> The raw concatenated array is ~1 GB and read **memory-mapped**, so every subject
> is cheap — no lazy streaming is needed (contrast the 49 GB wavelet cache). The
> wavelet file is opened only to read its small channel-name member.

## Configuration

In [ ]:
EXPERIMENT = ExperimentNames.ASSR
CONDITION = ConditionVariants.PLACEBO   # default per project convention
MUSIC_TYPE = MusicTypeVariants.ASSR

# ── Subjects to reduce ────────────────────────────────────────────────────
# None = ALL subjects (the raw array is memory-mapped, so every subject is cheap).
# Set to an explicit list of CONCATENATED_PERSON_INDEX values to subset.
SUBJECT_INDICES = None

# ── Stimulus-locked epoch window ──────────────────────────────────────────
# Pad kept BEFORE each onset (negative time), in seconds. The post-onset length
# is the shortest inter-onset interval (set in the subset cell), so longer
# intervals are trimmed and epochs never overlap.
PRE_PAD_S = 0.1

ASSR_FREQ = 40.0           # expected steady-state frequency (Hz)
SFREQ = 250.0              # sampling rate of the concatenated data

# Z-score each channel against the WHOLE recording before epoching, so every
# electrode has unit variance and contributes equally to the channel PCA
# (correlation-PCA). Set False to run PCA on raw µV (covariance-PCA, dominated by
# high-variance electrodes; the time course is then in µV).
ZSCORE_PER_CHANNEL = True

# ── Channel-name source ───────────────────────────────────────────────────
# The concatenated channel order is recovered from the wavelet cache's small
# `feature_names` member (only that member is read — the 49 GB data stream is
# never touched). Needed to map PC1 loadings onto the topomap montage order.
WAVELET_FREQ_SIG = "1.000_50.000_50"   # freqs[0]_freqs[-1]_n_freqs
N_WAVELET_FREQS = 50

# ── Plot saving ───────────────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "00-preprocessing"
    / "plots"
    / "assr_raw_pca"
    / f"{CONDITION.value}_{MUSIC_TYPE.value}"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Group          : {CONDITION.value} / {MUSIC_TYPE.value}")
print(f"Subjects       : {'ALL' if SUBJECT_INDICES is None else SUBJECT_INDICES}")
print(f"Pre-onset pad  : {PRE_PAD_S} s")
print(f"Z-score/channel: {ZSCORE_PER_CHANNEL}")
print(f"Plots -> {PLOTS_DIR}")

## Load Concatenated Raw Data, Onsets, Metadata & Channel Names

In [ ]:
safe_label = f"{CONDITION.value}_{MUSIC_TYPE.value}"

concat_dir = (
    ProjectPaths.PROCESSED_DATA_DIR
    / EXPERIMENT.value
    / PreprocessedDataVariants.CONCATENATED.value
)
raw_path = concat_dir / f"{safe_label}.npy"
onsets_path = concat_dir / f"{safe_label}{ProjectPaths.STIMULUS_ONSETS_SUFFIX}"
meta_path = concat_dir / f"{safe_label}.metadata.csv"

wavelet_path = (
    ProjectPaths.PROCESSED_DATA_DIR
    / EXPERIMENT.value
    / "wavelets"
    / "broadband"
    / f"{safe_label}__wavelet_power__{WAVELET_FREQ_SIG}__freqdim1.npz"
)
for p in (raw_path, onsets_path, meta_path, wavelet_path):
    assert p.exists(), f"Missing expected file: {p}"

# Memory-mapped raw array (~1 GB) — only the accessed subjects land in memory.
raw_mm = np.load(raw_path, mmap_mode="r")            # (n_subj, n_ch, n_times)
onsets = np.load(onsets_path)                        # (n_onsets,) shared sample idx
meta = pd.read_csv(meta_path, index_col=0)
n_subj_total, n_channels, n_times_raw = raw_mm.shape

# Channel names live in the wavelet feature_names (layout: channel*N_FREQS). Only
# that small member is decompressed; the concatenated raw shares the same order.
with zipfile.ZipFile(wavelet_path) as _z:
    with _z.open("feature_names.npy") as _f:
        feature_names = npformat.read_array(_f, allow_pickle=True)
channel_names = [str(feature_names[i * N_WAVELET_FREQS]).split("@")[0]
                 for i in range(len(feature_names) // N_WAVELET_FREQS)]
assert len(channel_names) == n_channels, (
    f"channel-name count {len(channel_names)} != raw n_channels {n_channels}"
)

gaps = np.diff(onsets)
print(f"Concatenated raw : {raw_mm.shape}  (dtype {raw_mm.dtype})")
print(f"Stimulus onsets  : {onsets.shape}  range [{onsets.min()}, {onsets.max()}]")
print(f"Inter-onset gap  : min {int(gaps.min())}, median {int(np.median(gaps))}, "
      f"max {int(gaps.max())} samples")
print(f"Channels parsed  : {n_channels} (first={channel_names[0]}, "
      f"last={channel_names[-1]})")
meta[["SingleDataMetadata.PARTICIPANT_ID", "SingleDataMetadata.CONDITION",
      "SingleDataMetadata.CONCATENATED_PERSON_INDEX"]].head()

## Subset Selection & Epoch Window

The post-onset window length is the **shortest inter-onset interval** (`gaps.min()`),
so every epoch fits between its onset and the next one. A fixed `PRE_PAD_S` pad is
kept before onset so the time course shows the negative-time baseline.

In [ ]:
subject_indices = (
    list(range(n_subj_total)) if SUBJECT_INDICES is None else list(SUBJECT_INDICES)
)

# Epoch window in samples: fixed pre-onset pad, post length = shortest gap.
PRE = int(round(PRE_PAD_S * SFREQ))
POST = int(gaps.min())                              # trim to shortest inter-onset gap
epoch_times = np.arange(-PRE, POST) / SFREQ         # (win,) seconds, t=0 at onset

# Map subject indices -> participant labels via the concatenated metadata.
pidx_col = "SingleDataMetadata.CONCATENATED_PERSON_INDEX"
pid_col = "SingleDataMetadata.PARTICIPANT_ID"
idx_to_pid = dict(zip(meta[pidx_col], meta[pid_col].astype(str).str.zfill(3)))
# Order subjects by participant ID so every per-participant plot is sorted by PID
# rather than by concatenated index.
subject_indices = sorted(
    subject_indices, key=lambda si: int(idx_to_pid.get(si, "9999"))
)
subject_labels = [f"PSI{idx_to_pid.get(si, '???')}" for si in subject_indices]

print(f"Selected subjects: {dict(zip(subject_indices, subject_labels))}")
print(f"Epoch window     : {PRE + POST} samples ({PRE} pre, {POST} post) "
      f"= [{epoch_times[0]:.3f}, {epoch_times[-1]:.3f}] s")

## Helper — stimulus-locked trial averaging

In [ ]:
def epoch_average(arr, onsets, pre, post):
    """Average fixed windows around each onset along the LAST axis.

    Args:
        arr: array whose last axis is time, e.g. ``(n_channels, n_times)``.
        onsets: stimulus onset sample indices.
        pre, post: samples kept before / after each onset (window = pre + post).

    Returns:
        ``(averaged, n_used)`` where the time axis is replaced by the
        ``pre + post`` window, averaged over every onset whose window fits inside
        the recording (edge windows are skipped).
    """
    n_time = arr.shape[-1]
    acc = None
    n_used = 0
    for o in onsets:
        s, e = o - pre, o + post
        if s < 0 or e > n_time:
            continue
        seg = arr[..., s:e]
        acc = seg.astype(np.float64) if acc is None else acc + seg
        n_used += 1
    if n_used == 0:
        raise ValueError("No onset window fits inside the recording.")
    return acc / n_used, n_used


print("Helper defined.")

## Trial-Average & Channel PCA — first-component per participant

For each subject: optionally z-score every channel against the whole recording
(equal electrode influence), epoch the signed signal around every onset and average
(the evoked response, `(n_channels, win)`), then fit a PCA where each **time
sample** is an observation and each **channel** a variable. The first component's
**score** is the component's time course `(win,)`; its **loading** is the
component's scalp topography `(n_channels,)`.

**Polarity alignment across participants.** A PCA component's sign is arbitrary, so
some participants come out with flipped polarity. Rather than fixing each subject
independently, every subject's loading is aligned to a common **template** (the
iteratively-refined group-mean loading) — forcing one consistent polarity across
participants — and the global orientation is then anchored to a **reference
channel** (`Cz` if present) so the sign is stable. The score is flipped with the
loading, so each subject's topography and time course stay consistent.

In [ ]:
time_courses = {}   # subject idx -> (win,) PC1 time course
loadings = {}       # subject idx -> (n_channels,) PC1 spatial pattern (topomap)
explained = {}      # subject idx -> PC1 explained variance ratio

n_used = None
for si in subject_indices:
    sig = np.asarray(raw_mm[si])                    # (n_ch, n_times)
    if ZSCORE_PER_CHANNEL:
        # Unit variance per channel over the WHOLE recording -> every electrode
        # contributes equally to the channel PCA (correlation-PCA).
        sig = zscore(sig, axis=1)
    evoked, n_used = epoch_average(sig, onsets, PRE, POST)  # (n_ch, win)
    # PCA over channels: observations = time samples, variables = channels.
    X = evoked.T                                    # (win, n_channels)
    pca = PCA(n_components=1)
    time_courses[si] = pca.fit_transform(X)[:, 0]   # (win,) PC1 time course
    loadings[si] = pca.components_[0]               # (n_channels,) spatial loading
    explained[si] = float(pca.explained_variance_ratio_[0])

# ── Align PC1 polarity ACROSS participants ────────────────────────────────
# A PCA component's sign is arbitrary, so some participants come out with flipped
# polarity. Align every subject's spatial loading to a common template (the
# iteratively-refined group-mean loading), which forces one consistent polarity
# across participants. Then anchor the whole set's global orientation with a
# reference channel so the sign is stable (independent of subject ordering).
subs = list(subject_indices)
loading_mat = np.array([loadings[si] for si in subs])   # (n_subj, n_channels)
template = loading_mat[0].copy()
for _ in range(10):
    flips = np.sign(loading_mat @ template)
    flips[flips == 0] = 1.0
    loading_mat = loading_mat * flips[:, None]
    template = loading_mat.mean(axis=0)
# Stable global orientation: make the reference channel's group loading positive.
ref_name = "Cz" if "Cz" in channel_names else channel_names[int(np.argmax(np.abs(template)))]
ref_idx = channel_names.index(ref_name)
if template[ref_idx] < 0:
    loading_mat, template = -loading_mat, -template
# Apply the aligned sign to BOTH the loading (topomap) and score (time course).
n_flipped = 0
for k, si in enumerate(subs):
    flip = np.sign(np.dot(loadings[si], loading_mat[k])) or 1.0
    if flip < 0:
        n_flipped += 1
    loadings[si] = loading_mat[k]
    time_courses[si] = time_courses[si] * flip

print(f"Trial-averaged {n_used} stimuli per participant; PCA over {n_channels} "
      f"channels.")
print(f"Polarity aligned across {len(subs)} participants "
      f"(reference channel {ref_name}; {n_flipped} subject(s) flipped).")
for si, label in zip(subject_indices, subject_labels):
    print(f"  {label}: PC1 explains {explained[si] * 100:.1f}% of channel variance")

### Per-Participant PC1 Time Course

In [ ]:
n_subj = len(subject_indices)
unit = "z-scored" if ZSCORE_PER_CHANNEL else "µV"
ncols = min(5, n_subj)
nrows = int(np.ceil(n_subj / ncols))

fig, axes = plt.subplots(
    nrows, ncols, figsize=(3.6 * ncols, 2.7 * nrows), sharex=True, squeeze=False
)
flat = axes.flatten()
for ax, si, label in zip(flat, subject_indices, subject_labels):
    ax.plot(epoch_times, time_courses[si], lw=1.5, color="C0")
    ax.axvline(0.0, color="red", ls="--", lw=0.8)
    ax.set_title(f"{label}  (PC1 {explained[si] * 100:.0f}%)", fontsize=9)
for ax in flat[n_subj:]:
    ax.axis("off")
fig.supxlabel("Time relative to onset (s)")
fig.supylabel(f"PC1 projected signal ({unit}, polarity-aligned)")
fig.suptitle(
    f"Per-participant channel-PCA first-component time course — "
    f"{CONDITION.value}/{MUSIC_TYPE.value} (n={n_subj})",
    y=1.0,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "raw_pca_timecourse_per_participant.png", dpi=150,
                bbox_inches="tight")
plt.show()

### PC1 Time Course — overlay + group mean

All participants overlaid with the across-participant mean. Polarity is now aligned
across participants (loadings matched to a common template), so the mean is a
meaningful summary wherever the dominant spatial mode is shared across subjects.

In [ ]:
unit = "z-scored" if ZSCORE_PER_CHANNEL else "µV"
fig, ax = plt.subplots(figsize=(11, 5))
for si, label in zip(subject_indices, subject_labels):
    ax.plot(epoch_times, time_courses[si], lw=1.1, alpha=0.75, label=label)
group_tc = np.mean([time_courses[si] for si in subject_indices], axis=0)
ax.plot(epoch_times, group_tc, lw=2.6, color="black", label="group mean")
ax.axvline(0.0, color="red", ls="--", lw=1, label="onset")
ax.set_title(
    f"Channel-PCA first-component time course — "
    f"{CONDITION.value}/{MUSIC_TYPE.value} (n={n_subj})"
)
ax.set_xlabel("Time relative to onset (s)")
ax.set_ylabel(f"PC1 projected signal ({unit}, polarity-aligned)")
ax.legend(loc="upper right", fontsize=7, ncol=3)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "raw_pca_timecourse_overlay.png", dpi=150)
plt.show()

## Topomap Montage — electrode positions

Electrode positions come from one `RAW_CROPPED` recording's `Info` (same channel
order guaranteed by remapping onto the canonical `channel_names`).

In [ ]:
topo_handler = DatasetHandler(EXPERIMENT, CoordinateSystems.HYDROGEL_257_NO_FIDUCIALS)
info_fname = meta.loc[
    meta[pidx_col] == subject_indices[0], "SingleDataMetadata.FILENAME"
].iloc[0]
topo_info = (
    topo_handler.load_data_file(
        info_fname,
        is_processed=True,
        processed_data_type=PreprocessedDataVariants.RAW_CROPPED,
        preload=False,
    )
    .pick("eeg")
    .info
)
name_pos = {n: i for i, n in enumerate(channel_names)}
assert set(topo_info["ch_names"]) <= set(name_pos), (
    "RAW_CROPPED channels are not a subset of the concatenated channel names."
)
info_order = [name_pos[n] for n in topo_info["ch_names"]]
print(f"Topomap montage: {len(topo_info['ch_names'])} electrodes "
      f"(reordered onto concatenated channel order).")

### Per-Participant PC1 Topomap

Scalp topography of the first component per participant (loading vector), following
the notebook-05 convention — signed values, diverging `RdBu_r`, symmetric colour
scale. Polarity is aligned across participants (loadings matched to a common
template), so topographies are directly comparable and the final group-average
panel is meaningful.

In [ ]:
panels = [(f"{subject_labels[k]}  (PC1 {explained[si] * 100:.0f}%)",
           loadings[si][info_order])
          for k, si in enumerate(subject_indices)]
group_loading = np.mean([loadings[si] for si in subject_indices], axis=0)
mean_ev = float(np.mean([explained[si] for si in subject_indices])) * 100
panels.append((f"group average  (mean PC1 {mean_ev:.0f}%)",
               group_loading[info_order]))

# Symmetric shared scale (nb05 convention): 99th percentile of |loading|.
vlim = float(np.percentile(np.abs(np.concatenate([v for _, v in panels])), 99))
if vlim == 0.0:
    vlim = 1e-12

n_panels = len(panels)
ncols = min(5, n_panels)
nrows = int(np.ceil(n_panels / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(2.7 * ncols, 2.9 * nrows),
                         squeeze=False)
flat = axes.flatten()
im = None
for ax, (label, vals) in zip(flat, panels):
    im, _ = mne.viz.plot_topomap(
        vals, topo_info, axes=ax, show=False, cmap="RdBu_r",
        vlim=(-vlim, vlim), contours=4,
    )
    ax.set_title(label, fontsize=9)
for ax in flat[n_panels:]:
    ax.axis("off")
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6, label="PC1 loading (a.u.)")
fig.suptitle(
    f"Per-participant channel-PCA first-component topography — "
    f"{CONDITION.value}/{MUSIC_TYPE.value} (n={n_subj})",
    y=1.0,
)
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "raw_pca_topomap_per_participant.png", dpi=150,
                bbox_inches="tight")
plt.show()